# Cynbit Discovery Copilot — POC walkthrough

**AI/ML Engineer Intern challenge — Cynbit Technologies.**

This notebook walks through the Proof of Concept end to end. It **runs in
Offline Mode by default** — zero API keys, zero network. An optional cell at
the end switches to live GLM inference if you have a free key.

> All example inputs are SYNTHETIC. Outputs are drafts for human review, never
> auto-sent to clients.

## 1. Setup
If running in Colab, first clone the repository and install deps:

In [ ]:
# Colab only (uncomment):
# !git clone https://github.com/<your-user>/cynbit-ai-opportunity.git
# %cd cynbit-ai-opportunity
# !pip install -q -r requirements.txt

import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from app.core.pipeline import DiscoveryPipeline, sanitize
from app.core.examples import DEMO_EXAMPLES
from app.core.prompts import PROMPT_VERSION

pipe = DiscoveryPipeline(provider="offline")  # deterministic, no network
print("prompt version:", PROMPT_VERSION)

## 2. Step 1 — the sanitizer (pure Python, no LLM)

Every input is treated as untrusted: PII is masked before anything downstream,
prompt-injection patterns are flagged, language and length are checked.

In [ ]:
demo_injection = DEMO_EXAMPLES[7]["text"]
print(demo_injection[:200], "...\n")

san = sanitize(demo_injection)
print("injection suspected:", san.injection_suspected)
print("patterns:", san.injection_patterns)
print("delimited head:", san.delimited_text[:80])

In [ ]:
san2 = sanitize("Reach me at someone@example.com or +91 98765 43210.")
print("PII masked:", san2.pii_masked, "| types:", san2.pii_types)
print(san2.sanitized_text)

## 3. The full pipeline (Offline Mode)

Pipeline: **sanitize → classify+extract → ground (catalog evidence) →
deterministic estimate → advise → validate**.

The badge below always shows which engine produced the output —
`offline_cached` replays a REAL precomputed GLM response for built-in
examples; `offline_rules` is the deterministic engine.

In [ ]:
brief = pipe.run(DEMO_EXAMPLES[0]["text"])   # exact demo input -> cached live-GLM output
print("mode:", brief.provenance.mode, "| cache hit:", brief.provenance.cache_hit)
print("services:", [(m.service_id, m.confidence) for m in brief.services])
print("complexity:", brief.estimate.complexity, brief.estimate.effort_range_weeks, "weeks")
print("top question:", brief.clarifying_questions[0].question)

In [ ]:
brief2 = pipe.run("We run a dental clinic in Jaipur and want patients to book "
                     "appointments online with WhatsApp reminders. Budget around "
                     "Rs 2.5 lakh, needed before Diwali.")
print("mode:", brief2.provenance.mode)   # arbitrary input -> deterministic engine
print("services:", [m.service_id for m in brief2.services])
print("extraction:", {k: brief2.extraction.model_dump()[k] for k in ("goal","budget_signal","timeline_signal","industry")})
print("complexity:", brief2.estimate.complexity if brief2.estimate else None)

### 3.1 Out-of-scope input — graceful redirect, nothing invented

In [ ]:
oos = pipe.run(DEMO_EXAMPLES[6]["text"])     # internship application
print("out_of_scope:", oos.out_of_scope)
print("services:", oos.services)
print("suggested response:", (oos.suggested_response or "")[:140])

### 3.2 Prompt-injection attempt — flagged, stays on task, no compliance

In [ ]:
inj = pipe.run(DEMO_EXAMPLES[7]["text"])
print("flagged:", inj.sanitization.injection_suspected)
print("underlying request still classified:", [m.service_id for m in inj.services])
advice = " ".join(r.risk + " " + q.question for r in inj.risks for q in inj.clarifying_questions[:1])
assert "90% discount" not in advice.lower()
print("no injected instructions complied with ✓")

## 4. Evaluation snapshot (held-out synthetic split)

Full methodology and per-example predictions: `evals/`. Run locally:
`python evals/run_eval.py` (offline) or `--provider glm` (live).

In [ ]:
import json, pathlib
results = pathlib.Path(ROOT) / "evals" / "results"
for f in sorted(results.glob("metrics_*.json")):
    m = json.loads(f.read_text())
    print(f"{f.stem.replace('metrics_',''):<22} exact-set={m['service_exact_set_acc']:<6} "
          f"macroF1={m['service_macro_f1']:<6} cx={m['complexity_acc']:<6} "
          f"inj={'n/a' if m['injection_pass_rate'] is None else m['injection_pass_rate']}")

## 5. Optional — live GLM mode

Uncomment and set a free key from https://open.bigmodel.cn (CN) or
https://z.ai (international). The pipeline is identical; only steps 2 and 5
switch to schema-validated GLM calls with automatic fallback to Offline Mode.

In [ ]:
# import os
# os.environ["GLM_API_KEY"] = "your-free-key"           # never commit this
# os.environ["GLM_BASE_URL"] = "https://open.bigmodel.cn/api/paas/v4"
# os.environ["GLM_MODEL"] = "glm-4-flash"
#
# from app.core.llm_client import GLMClient
# live_pipe = DiscoveryPipeline(provider="glm", llm=GLMClient())
# b = live_pipe.run(DEMO_EXAMPLES[1]["text"])
# print("mode:", b.provenance.mode, "| latency:", b.provenance.latency_ms, "ms")
# print("tokens:", b.provenance.prompt_tokens, "/", b.provenance.completion_tokens)

## 6. Takeaways

1. **Pipeline, not a mega-prompt** — sanitizer, classifier, grounder,
   estimator, advisor are each testable; the estimate step is deterministic
   in every mode so pricing is never hallucinated.
2. **Grounding over fluency** — service matches carry catalog evidence with
   source URLs; unknowns become clarifying questions.
3. **Offline Mode is real** — cached live outputs + a deterministic engine
   mean the demo works with zero keys, and the badge never lets you mistake
   one for the other.

Limitations (honest): all numbers are on synthetic data; live metrics in the
repo were produced through the build environment's GLM gateway (served
glm-4-plus); complexity bands are coarse by design.